# Visualize discharge data

The following notebook will generate map visualizations of discharge data for all reach-level and basin-level algorithms.

In [ ]:
import datetime
import json
import pathlib
import warnings

import branca.colormap as cm
import folium
import geopandas as gpd
import netCDF4 as nc
import numpy as np
import pandas as pd
import shapely

In [ ]:
# Constants that require updating based on execution
REACHES_OF_INTEREST_FILE = pathlib.Path.cwd().joinpath("data").joinpath("reaches_sit.json")
PRIOR_SOS_GRANULE = pathlib.Path.cwd().joinpath("data").joinpath("na_sword_v16_SOS_priors.nc")
RESULT_SOS_GRANULE = pathlib.Path.cwd().joinpath("data").joinpath("na_sword_v16_SOS_results.nc")

In [ ]:
# Constants to capture algorithms and variables
FLPE = {
    "metroman": "allq",
    "momma": "Q",
    "neobam": "q",
    "sad": "Qa",
    "sic4dvar": "Q_da"
}
MOI = {
    "metroman": "q",
    "momma": "q",
    "neobam": "q",
    "sad": "q",
    "sic4dvar": "q"
}
FILL = {
    "f8": -999999999999.0,
    "i4": -999,
    "i8": -999999999999,
    "S1": "x",
    "S48": b"\x00" * 48        # 48-byte padded string fill (e.g., tile_name)
}

In [ ]:
# Retrieve reach identifiers used in processing
with open(REACHES_OF_INTEREST_FILE) as jf:
    reach_identifiers = json.load(jf)
reach_identifiers = [ int(reach_id) for reach_id in reach_identifiers ]
reach_identifiers.sort()

In [ ]:
# Open SOS files to work with underlying data
results = nc.Dataset(RESULT_SOS_GRANULE, format="NETCDF4")

In [ ]:
# Function to retrieve discharge
def retrieve_discharge(reach_id, reach_index, results, track_results):
    """Retrieve discharge from SOS results."""

    found_data = []
    
    # Reach-level algorithms
    for algorithm, variable in FLPE.items():
        if algorithm == "neobam":
            track_results[reach_id][algorithm] = results[algorithm][variable][variable][reach_index][0]
        else:
            track_results[reach_id][algorithm] = results[algorithm][variable][reach_index][0]

        # Set missing values to NaN
        missing = np.where(track_results[reach_id][algorithm] == FILL["f8"])
        track_results[reach_id][algorithm][missing] = np.nan

        if not np.isnan(track_results[reach_id][algorithm]).all():
            found_data.append(True)
        else:
            found_data.append(False)

        # Locate the mean value of the discharge time series
        track_results[reach_id][algorithm] = np.nanmean(track_results[reach_id][algorithm])

    # Basin-level algorithms
    track_results[reach_id]["moi"] = {}
    for algorithm, variable in MOI.items():
        track_results[reach_id]["moi"][algorithm] = results["moi"][algorithm][variable][reach_index][0]
        missing = np.where(track_results[reach_id]["moi"][algorithm] == FILL["f8"])
        track_results[reach_id]["moi"][algorithm][missing] = np.nan
    
        if not np.isnan(track_results[reach_id]["moi"][algorithm]).all():
            found_data.append(True)
        else:
            found_data.append(False)

        # Locate the mean value of the discharge time series
        track_results[reach_id]["moi"][algorithm] = np.nanmean(track_results[reach_id]["moi"][algorithm])

    return np.array(found_data)

In [ ]:
# Retrieve SWOT observation times
track_results = {}
plot_results = {}
results_with_data = 0
results_with_data_more_than_half = 0
for reach_id in reach_identifiers:
    
    # Locate index of reach
    reach_index = np.where(results['reaches']['reach_id'][:] == reach_id)

    # Locate algorithm results for data indexes
    track_results[reach_id] = {}
    found_data = retrieve_discharge(reach_id, reach_index, results, track_results)
    if found_data.any():
        results_with_data += 1
        plot_results[reach_id] = track_results[reach_id]
    if np.count_nonzero(found_data) >= found_data.size / 2:
        results_with_data_more_than_half += 1

print("Number of reach identifiers found: ", len(track_results.keys()))
print("Number of results with data: ", results_with_data)
print("Number of results where more than half the algorithms produced data: ", results_with_data_more_than_half)

In [ ]:
# Save results data
with open("plot_results.json", "w") as jf:
    json.dump(plot_results, jf, indent=2)

In [ ]:
# Close SOS files
results.close()

In [ ]:
# Build a dictionary that has one pandas dataframe per algorithm with reach_id, discharge columns
reach_ids = np.array(list(plot_results.keys()))

# FLPE
metroman = np.array([ reach["metroman"] for reach in plot_results.values() ])
momma = np.array([ reach["momma"] for reach in plot_results.values() ])
neobam = np.array([ reach["neobam"] for reach in plot_results.values() ])
sad = np.array([ reach["sad"] for reach in plot_results.values() ])
sic4dvar = np.array([ reach["sic4dvar"] for reach in plot_results.values() ])
# MOI
moi_metroman = np.array([ reach["moi"]["metroman"] for reach in plot_results.values() ])
moi_momma = np.array([ reach["moi"]["momma"] for reach in plot_results.values() ])
moi_neobam = np.array([ reach["moi"]["neobam"] for reach in plot_results.values() ])
moi_sad = np.array([ reach["moi"]["sad"] for reach in plot_results.values() ])
moi_sic4dvar = np.array([ reach["moi"]["sic4dvar"] for reach in plot_results.values() ])

# Pandas DF
flpe_dict = {}
moi_dict = {}
for algo, flpe, moi in (("metroman", metroman, moi_metroman), ("momma", momma, moi_momma), ("neobam", neobam, moi_neobam), ("sad", sad, moi_sad), ("sic4dvar", sic4dvar, moi_sic4dvar)):
    flpe_dict[algo] = pd.DataFrame({
        "reach_id": reach_ids,
        "discharge": flpe
    })
    moi_dict[algo] = pd.DataFrame({
        "reach_id": reach_ids,
        "discharge": moi
    })

In [ ]:
# Join discharge to GeoPandas DataFrame and extract discharge and geometries
flpe_gdf = {}
for algo, df in flpe_dict.items():
    # Read in SWORD as Geopandas dataframe
    sword = pathlib.Path.cwd().joinpath("data").joinpath("na_sword_reaches_hb74_v16.shp")
    gdf = gpd.read_file(sword)

    # Locate reach identifiers for river of interest
    reach_ids = np.array(list(plot_results.keys()))
    reach_mask = gdf["reach_id"].isin(reach_ids)
    gdf = gdf[reach_mask]
    
    gdf = gdf.join(df.set_index("reach_id"), on="reach_id")
    gdf["discharge"] = gdf["discharge"].fillna(FILL["f8"])
    flpe_gdf[algo] = gdf

moi_gdf = {}
for algo, df in moi_dict.items():
    # Read in SWORD as Geopandas dataframe
    sword = pathlib.Path.cwd().joinpath("data").joinpath("na_sword_reaches_hb74_v16.shp")
    gdf = gpd.read_file(sword)

    # Locate reach identifiers for river of interest
    reach_ids = np.array(list(plot_results.keys()))
    reach_mask = gdf["reach_id"].isin(reach_ids)
    gdf = gdf[reach_mask]
    
    gdf = gdf.join(df.set_index("reach_id"), on="reach_id")
    gdf["discharge"] = gdf["discharge"].fillna(FILL["f8"])
    moi_gdf[algo] = gdf

In [ ]:
# Function to visualize discharge on maps
def visualize_discharge(gdf):
    
    # Create map
    max_x = np.median(gdf["x"])
    max_y = np.median(gdf["y"])
    m = folium.Map([max_y, max_x], zoom_start=10, tiles="cartodb positron")
    
    # Create a color map
    arr = gdf["discharge"]
    arr = arr[arr != FILL["f8"]]    # filter out fill value for color
    min_d = np.min(arr)
    if np.isnan(min_d):
        min_d = FILL["f8"]
    max_d = np.max(gdf["discharge"])
    color_map = cm.LinearColormap(["blue", "green", "yellow", "red"], vmin=min_d, vmax=max_d)
    color_map.add_to(m)
    
    # Create a tool tip to display reach identifier and discharge values
    tooltip = folium.GeoJsonTooltip(
        fields=["reach_id", "discharge"],
        aliases=["Reach Identifier:", "Mean Discharge:"],
        sticky=False,
        labels=True,
        style="""
            background-color: #F0EFEF;
            border: 2px solid black;
            border-radius: 3px;
            box-shadow: 3px;
        """,
        max_width=800
    )

    # Style function for map
    def style_function(feature):
        value = feature["properties"]["discharge"]
        if value == FILL["f8"]:
            fill_color = "black"
        else:
            fill_color = color_map(value)
        return {
            "color": fill_color,
            "weight": 3
        }
    
    # Visualize mean discharge
    folium.GeoJson(
        gdf,
        style_function=style_function,
        tooltip=tooltip
    ).add_to(m)

    return m

In [ ]:
# Visualize and save maps
for algo, data in flpe_gdf.items():
    m = visualize_discharge(data)
    m_file = pathlib.Path.cwd().joinpath("data").joinpath("plots").joinpath(f"data_visualization_{algo}.html")
    m_file.parent.mkdir(parents=True, exist_ok=True)
    m.save(m_file)

for algo, data in moi_gdf.items():
    m = visualize_discharge(data)
    m_file = pathlib.Path.cwd().joinpath("data").joinpath("plots").joinpath(f"data_visualization_moi_{algo}.html")
    m_file.parent.mkdir(parents=True, exist_ok=True)
    m.save(m_file)